# Rocket Lab - Atividade 1
<br>

## Ingestão para Bronze

Primeiro notebook da pipeline da atividade, com os seguintes objetivos:
- criar o schema e as tabelas da camada Bronze;
- ler os arquivos CSV brutos do dataset Olist;
- salvar os dados no formato Delta na camada Bronze;
- ingerir a cotação do dólar via API do Banco Central.

Nesta etapa, os dados serão carregados de forma bruta do volume previamente criado.

___

#### Etapa de Imports, definições que o notebook vai usar e criação e configurações da camada Bronze
___

In [0]:
# Célula somente para imports
from pyspark.sql import functions as F
import requests

from datetime import datetime # Usado na verificação das datas de início e fim

In [0]:
# Criação schema da camada Bronze
spark.sql("CREATE CATALOG IF NOT EXISTS ecommerce_olist")
spark.sql("CREATE SCHEMA IF NOT EXISTS ecommerce_olist.bronze")


spark.sql("SHOW CATALOGS").show()
spark.sql("SHOW SCHEMAS IN ecommerce_olist").show()

+---------------+
|        catalog|
+---------------+
|ecommerce_olist|
|        samples|
|         system|
|      workspace|
+---------------+

+------------------+
|      databaseName|
+------------------+
|            bronze|
|           default|
|              gold|
|information_schema|
|           landing|
|            silver|
+------------------+



In [0]:
# Célula apenas para confirmar a utilização do catálogo e schema certos
spark.sql("USE CATALOG ecommerce_olist")
spark.sql("USE SCHEMA bronze")

DataFrame[]

In [0]:
# Definições dos caminhos e nomes dos arquivos e tabelas
volume_path = "/Volumes/ecommerce_olist/landing/dados_olist"

catalogo = "ecommerce_olist"
schema = "bronze"

# Mapeamento arquivo.csv -> nome da tabela
arquivos_tabelas = {
    "olist_customers_dataset.csv": "tb_customers",
    "olist_geolocation_dataset.csv": "tb_geolocalizacao",
    "olist_order_items_dataset.csv": "tb_order_items",
    "olist_order_payments_dataset.csv": "tb_order_payments",
    "olist_order_reviews_dataset.csv": "tb_order_reviews",
    "olist_orders_dataset.csv": "tb_orders",
    "olist_products_dataset.csv": "tb_products",
    "olist_sellers_dataset.csv": "tb_sellers",
    "product_category_name_translation.csv": "tb_product_category_name_translation"
}

# Apenas mostra os arquivos que estão no volume
display(dbutils.fs.ls(volume_path))

path,name,size,modificationTime
dbfs:/Volumes/ecommerce_olist/landing/dados_olist/olist_customers_dataset.csv,olist_customers_dataset.csv,11828674,1775441883000
dbfs:/Volumes/ecommerce_olist/landing/dados_olist/olist_geolocation_dataset.csv,olist_geolocation_dataset.csv,61273883,1775441889000
dbfs:/Volumes/ecommerce_olist/landing/dados_olist/olist_order_items_dataset.csv,olist_order_items_dataset.csv,15438671,1775441883000
dbfs:/Volumes/ecommerce_olist/landing/dados_olist/olist_order_payments_dataset.csv,olist_order_payments_dataset.csv,5777138,1775441881000
dbfs:/Volumes/ecommerce_olist/landing/dados_olist/olist_order_reviews_dataset.csv,olist_order_reviews_dataset.csv,14451670,1775441883000
dbfs:/Volumes/ecommerce_olist/landing/dados_olist/olist_orders_dataset.csv,olist_orders_dataset.csv,17654914,1775441884000
dbfs:/Volumes/ecommerce_olist/landing/dados_olist/olist_products_dataset.csv,olist_products_dataset.csv,3458105,1775441881000
dbfs:/Volumes/ecommerce_olist/landing/dados_olist/olist_sellers_dataset.csv,olist_sellers_dataset.csv,249900,1775441880000
dbfs:/Volumes/ecommerce_olist/landing/dados_olist/product_category_name_translation.csv,product_category_name_translation.csv,2613,1775441880000


In [0]:
# Célula destinada a declaração e desenvolvimento de funções úteis ao notebook

# Função de validação de data
def validar_formato_data(data: str, nome_param: str):
    try:
        datetime.strptime(data, "%m-%d-%Y")
    except ValueError:
        raise ValueError(f"Parâmetro '{nome_param}' com formato inválido: '{data}'. Esperado: MM-DD-AAAA")


# Função de leitura dos arquivos e criação das tabelas
def ingestao_csv(nome_arquivo, nome_tabela):
    try:
        caminho_arquivo = f"{volume_path}/{nome_arquivo}"
        nome_tabela_completo = f"{catalogo}.{schema}.{nome_tabela}"

        df = (
            spark.read
            .option("header", True)
            .option("sep", ",")
            .option("inferSchema", False)
            .option("multiLine", True)
            .option("quote", '"')
            .option("escape", '"')
            .csv(caminho_arquivo)
        )

        # Validação
        if df.count() == 0:
            raise ValueError(f"O arquivo {nome_arquivo} está vazio ou não pôde ser lido.")

        # Adiciona timestamp de ingestão
        df_com_hora_insercao = df.withColumn("timestamp_ingestion", F.current_timestamp())

        # Escrita no formato Delta
        df_com_hora_insercao.write \
            .format("delta") \
            .mode("overwrite") \
            .option("overwriteSchema", "true") \
            .saveAsTable(nome_tabela_completo)

        print(f"Tabela {nome_tabela_completo} criada com sucesso.")

    except Exception as e:
        print(f"Erro ao processar {nome_arquivo}: {str(e)}")

In [0]:
# Definição de data início e fim e habilitando mudanças como parâmetros do notebook
dbutils.widgets.text("data_inicio", "03-01-2016") # Escolhi essas datas como datas "padrões" pelas datas informadas no dataset constarem nos anos de 2016 e 2018
dbutils.widgets.text("data_fim", "12-31-2018") 
data_inicio = dbutils.widgets.get("data_inicio")
data_fim = dbutils.widgets.get("data_fim")

validar_formato_data(data_inicio, "data_inicio")
validar_formato_data(data_fim, "data_fim")
print("Data início:", data_inicio)
print("Data fim:", data_fim)

Data início: 03-01-2016
Data fim: 12-31-2018


#### Etapa onde serão realizadas execuções para construir a camada Bronze
___

In [0]:
# Laço que vai ler todos os arquivos e criar todas as tabelas
for arquivo, tabela in arquivos_tabelas.items():
    ingestao_csv(arquivo, tabela)

Tabela ecommerce_olist.bronze.tb_customers criada com sucesso.
Tabela ecommerce_olist.bronze.tb_geolocalizacao criada com sucesso.
Tabela ecommerce_olist.bronze.tb_order_items criada com sucesso.
Tabela ecommerce_olist.bronze.tb_order_payments criada com sucesso.
Tabela ecommerce_olist.bronze.tb_order_reviews criada com sucesso.
Tabela ecommerce_olist.bronze.tb_orders criada com sucesso.
Tabela ecommerce_olist.bronze.tb_products criada com sucesso.
Tabela ecommerce_olist.bronze.tb_sellers criada com sucesso.
Tabela ecommerce_olist.bronze.tb_product_category_name_translation criada com sucesso.


In [0]:
# Apenas um check para validar a construção das tabelas
for tabela in arquivos_tabelas.values():
    qtd = spark.table(f"{catalogo}.{schema}.{tabela}").count()
    print(f"{tabela}: {qtd} registros")

tb_customers: 99441 registros
tb_geolocalizacao: 1000163 registros
tb_order_items: 112650 registros
tb_order_payments: 103886 registros
tb_order_reviews: 99224 registros
tb_orders: 99441 registros
tb_products: 32951 registros
tb_sellers: 3095 registros
tb_product_category_name_translation: 71 registros


In [0]:
# Célula para visualização de alguma tabela
display(spark.table("ecommerce_olist.bronze.tb_customers").limit(5))

customer_id,customer_unique_id,customer_zip_code_prefix,customer_city,customer_state,customer_name,customer_gender,customer_birth_date,customer_age,timestamp_ingestion
06b8999e2fba1a1fbc88172c00ba8bc7,861eff4711a542e4b93843c6dd7febb0,14409,franca,SP,Zoe Nogueira,F,1956-09-07,70,2026-04-07T20:24:18.288Z
18955e83d337fd6b2def6b18a428ac77,290c77bc529b7ac935b93aa66c333dc3,9790,sao bernardo do campo,SP,Liam Viana,M,1974-09-23,52,2026-04-07T20:24:18.288Z
4e7b3e00288586ebd08712fdd0374a03,060e732b5b29e8181a18229c7b0b2b5e,1151,sao paulo,SP,Diego Aparecida,M,1964-05-20,62,2026-04-07T20:24:18.288Z
b2b6027bc5c5109e529d4dc6358b12c3,259dac757896d24d7702b9acbbff3f3c,8775,mogi das cruzes,SP,Marcos Vinicius Azevedo,M,1967-01-14,59,2026-04-07T20:24:18.288Z
4f2d8ab171c80ec8364f7c12e35b23ad,345ecd01c38d18a9036ed96c73b8d066,13056,campinas,SP,Srta. Juliana Siqueira,F,1950-08-01,76,2026-04-07T20:24:18.288Z


#### Etapa de extração e armazenamento dos dados da API
___

In [0]:
# Extração dos dados da API do banco central
url = (
    "https://olinda.bcb.gov.br/olinda/servico/PTAX/versao/v1/odata/"
    f"CotacaoDolarPeriodo(dataInicial=@dataInicial,dataFinalCotacao=@dataFinalCotacao)?"
    f"@dataInicial='{data_inicio}'&@dataFinalCotacao='{data_fim}'"
    "&$select=dataHoraCotacao,cotacaoCompra&$format=json"
)

response = requests.get(url)
response.raise_for_status()

dados_api = response.json()["value"]

# Levantamento de erro caso o intervalo de datas escolhido não seja válido
if not dados_api:
    raise ValueError(f"API não retornou cotações para o período {data_inicio} a {data_fim}. Verifique se o intervalo contém dias úteis.")

print(f"Quantidade de registros retornados pela API: {len(dados_api)}")

Quantidade de registros retornados pela API: 711


In [0]:
# Organização dos dados da API para um df e criação da tabela de cotações
df_dolar = spark.createDataFrame(dados_api)
df_dolar = df_dolar.withColumn("timestamp_ingestion", F.current_timestamp()) # Adição do timestamp do momento da inserção dos dados

df_dolar.write.format("delta").mode("overwrite").option("overwriteSchema", "true").saveAsTable(f"{catalogo}.{schema}.tb_cotacao_dolar")
print("Tabela ecommerce_olist.bronze.tb_cotacao_dolar criada com sucesso.")

Tabela ecommerce_olist.bronze.tb_cotacao_dolar criada com sucesso.


In [0]:
# Apenas a visualização da tabela de cotações
display(
    spark.table("ecommerce_olist.bronze.tb_cotacao_dolar")
    .orderBy(F.col("dataHoraCotacao").desc())
    .limit(5)
)

cotacaoCompra,dataHoraCotacao,timestamp_ingestion
3.8742,2018-12-31 11:04:06.627,2026-04-07T20:24:51.685Z
3.8742,2018-12-28 13:03:50.793,2026-04-07T20:24:51.685Z
3.9324,2018-12-27 13:07:19.149,2026-04-07T20:24:51.685Z
3.9252,2018-12-26 13:05:34.012,2026-04-07T20:24:51.685Z
3.8839,2018-12-24 11:05:11.875,2026-04-07T20:24:51.685Z
